# 02. Реальный Jenkins connector

Добавляем первый внешний DevOps connector. Показываем три режима: dry-run preview, read-only metadata и gated real build.


![Jenkins and VK bridge](../visuals/openclaw_path/05_jenkins_vk_cron.svg)

Jenkins — хороший пример real connector: читать metadata можно безопасно, запуск build должен быть явно разрешён.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

def write_text(path: str, content: str) -> Path:
    target = REPO_ROOT / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return target

def write_json(path: str, payload: dict) -> Path:
    target = REPO_ROOT / path
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n')
    return target


In [ ]:
from connectors.jenkins import get_jenkins_job_info, trigger_jenkins_job

print(trigger_jenkins_job.invoke({
    "parameters": {"OPENCLAW_SMOKE": "true"},
    "dry_run": True,
}))


## Live metadata

Эта ячейка делает реальный read-only HTTP-запрос к Jenkins job. Если сеть/VPN недоступны, dry-run выше всё равно показывает правильную форму запроса.


In [ ]:
print(get_jenkins_job_info.invoke({}))


## Real build, только через gate

Запуск build включается только переменной `OPENCLAW_RUN_REAL_JENKINS_PIPELINE=1`.


In [ ]:
if os.getenv("OPENCLAW_RUN_REAL_JENKINS_PIPELINE") == "1":
    print(trigger_jenkins_job.invoke({
        "parameters": {"OPENCLAW_SMOKE": "true"},
        "dry_run": False,
    }))
else:
    print("Skipped real Jenkins build. Set OPENCLAW_RUN_REAL_JENKINS_PIPELINE=1 to enable it.")


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\n\nfrom connectors.jenkins import JENKINS_TOOLS\n\nload_dotenv()\n\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\n\ndef _backend():\n    from deepagents.backends import FilesystemBackend\n\n    return FilesystemBackend(root_dir=Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).resolve(), virtual_mode=True)\n\n\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw with a Jenkins connector. Use dry-run before triggering builds.\nRead Jenkins metadata when useful, and explain network failures plainly.\n\"\"\"\n\nagent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=JENKINS_TOOLS,\n    system_prompt=BASE_PROMPT,\n    backend=_backend(),\n)\n"

CONFIG = {
    "dependencies": ["."],
    "graphs": {"openclaw_path_02": "./agents/openclaw_path_02_jenkins.py:agent"},
    "env": ".env",
}

print(write_text('agents/openclaw_path_02_jenkins.py', ENTRYPOINT).relative_to(REPO_ROOT))
print(write_json('langgraph.openclaw_path.json', CONFIG).relative_to(REPO_ROOT))


## Проверочный prompt

```text
Use the Jenkins connector. First show a dry-run build request with OPENCLAW_SMOKE=true, then read the configured job metadata.
```
